    Introduction

A film that is financially successful may not be critically successful and vice versa. What is then the relation between the critical success of the movie and the financial one? Do movies that achieved a financial success tend to also achieve a critical success?

How do I understand these two types of success?

1. Financial Success: overall, how much money did the movie make and the ROI; 

2. Critical Success: how well reviewed was the movie - that includes, for example the IMDB scores. 

In the second case of success there is a following problem: there might be movies that have 10/10 average rating score, but only 5 people rated it in total. If we only took ratings by themselves, this movie would be more critically successful than an established classic with score of only 9,5, but hundreds of thousands of reviews. In order to fix this I will use a formula provided by IMDb available here: https://help.imdb.com/article/imdb/track-movies-tv/ratings-faq/G67Y87TFYYP6TWAV#  under ‘How do you calculate the rank of movies and TV shows on the Top 250 Movies and Top 250 TV Show charts?’

This formula takes into account the number of ratings each title has received, minimum ratings required to be on the list, and the mean rating for all titles:

weighted rating (WR) = (v ÷ (v+m)) x R + (m ÷ (v+m)) x C

Where:

R = mean rating for all titles

v = number of ratings for the title

m = minimum ratings required to be listed in the Top Rated 250 chart

C = the mean rating across the whole report

    Analysis

0. Loading the necessary libraries

In [1]:
import pandas as pd
import numpy as np
import duckdb
import json
import scipy.stats

1. Exploring the ratings dataset

First, I'll look at the movie ratings data.

In [ ]:
rating = pd.read_csv('data/movie_data.csv')

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/bartomiejowsiany/movie-ratings-data/movie_data.csv'

In [ ]:
rating.head()

,index,director_name,duration,actor_2_name,genres,actor_1_name,movie_title,num_voted_users,actor_3_name,movie_imdb_link,num_user_for_reviews,language,country,title_year,imdb_score
0,0,James Cameron,178.0,Joel David Moore,Action|Adventure|Fantasy|Sci-Fi,CCH Pounder,Avatar,886204,Wes Studi,http://www.imdb.com/title/tt0499549/?ref_=fn_t...,3054.0,English,USA,2009.0,7.9
1,1,Gore Verbinski,169.0,Orlando Bloom,Action|Adventure|Fantasy,Johnny Depp,Pirates of the Caribbean: At World's End,471220,Jack Davenport,http://www.imdb.com/title/tt0449088/?ref_=fn_t...,1238.0,English,USA,2007.0,7.1
2,2,Sam Mendes,148.0,Rory Kinnear,Action|Adventure|Thriller,Christoph Waltz,Spectre,275868,Stephanie Sigman,http://www.imdb.com/title/tt2379713/?ref_=fn_t...,994.0,English,UK,2015.0,6.8
3,3,Christopher Nolan,164.0,Christian Bale,Action|Thriller,Tom Hardy,The Dark Knight Rises,1144337,Joseph Gordon-Levitt,http://www.imdb.com/title/tt1345836/?ref_=fn_t...,2701.0,English,USA,2012.0,8.5
4,4,Doug Walker,NaN,Rob Walker,Documentary,Doug Walker,Star Wars: Episode VII - The Force Awakens ...,8,NaN,http://www.imdb.com/title/tt5289954/?ref_=fn_t...,NaN,NaN,NaN,NaN,7.1


In [ ]:
rating.columns

Index(['index', 'director_name', 'duration', 'actor_2_name', 'genres',
       'actor_1_name', 'movie_title', 'num_voted_users', 'actor_3_name',
       'movie_imdb_link', 'num_user_for_reviews', 'language', 'country',
       'title_year', 'imdb_score'],
      dtype='object')

In [ ]:
rating.shape

(5043, 15)

As we can see, our data set contains 15 columns and 5043 rows. For our analysis we will only need three columns: 'movie_title', 'imdb_score' and 'num_user_for_reviews'.

In [ ]:
ratings = rating[['movie_title','imdb_score','num_user_for_reviews']].copy()
ratings.head()

,movie_title,imdb_score,num_user_for_reviews
0,Avatar,7.9,3054.0
1,Pirates of the Caribbean: At World's End,7.1,1238.0
2,Spectre,6.8,994.0
3,The Dark Knight Rises,8.5,2701.0
4,Star Wars: Episode VII - The Force Awakens ...,7.1,NaN


2. Cleaing the dataset

In [ ]:
ratings.isnull().sum()

movie_title              0
imdb_score               0
num_user_for_reviews    21
dtype: int64

It is worth noting that we already see some possible flaws. Even though, there are only 21 rows containing null values, already in top 5 we can see that big name like Star Wars: Episode VII - The Force Awakens lacks the number of reviews, which might inflence the outcome of the analysis.

In [ ]:
ratings[ratings.isnull().any(axis=1)]

,movie_title,imdb_score,num_user_for_reviews
4,Star Wars: Episode VII - The Force Awakens ...,7.1,NaN
279,"10,000 B.C.",7.2,NaN
1510,Black Water Transit,7.2,NaN
2342,The Doombolt Chase,7.2,NaN
2345,The Touch,5.2,NaN
2370,"Gone, Baby, Gone",6.6,NaN
2765,Towering Inferno,9.5,NaN
2870,Del 1 - Män som hatar kvinnor,8.1,NaN
3816,Running Forever,8.6,NaN
4323,"To Be Frank, Sinatra at 100",7.4,NaN


Fortunately, due to low number of those null values we can see that the rest of titles are not as influential as Star Wars (no offence to A Beginner's Guide to Snuff and others), so the outcome might not be as affected after all.

In [ ]:
ratings_clean = ratings.dropna()

In [ ]:
ratings_clean.loc[:, 'movie_title'] = ratings_clean['movie_title'].str.strip()

In [ ]:
ratings_clean.duplicated().sum()

np.int64(122)

As we can see, there are 122 duplicate rows that also need to be cleaned.

In [ ]:
ratings_clean = ratings_clean.drop_duplicates()
f'Number of duplicate rows: {ratings_clean.duplicated().sum()}'

'Number of duplicate rows: 0'

3. Analysis of the ratings data set

Now I will use some basic SQL to see the top 100 movie titles by the number of reviews. This allows to see the higher range of the number of ratings.

In [ ]:
duckdb.query("""
SELECT * 
FROM ratings_clean
ORDER BY num_user_for_reviews DESC
             LIMIT 100""").to_df()

,movie_title,imdb_score,num_user_for_reviews
0,The Lord of the Rings: The Fellowship of the Ring,8.8,5060.0
1,The Dark Knight,9.0,4667.0
2,The Shawshank Redemption,9.3,4144.0
3,The Matrix,8.7,3646.0
4,Star Wars: Episode I - The Phantom Menace,6.5,3597.0
...,...,...,...
95,X-Men,7.4,1401.0
96,Forrest Gump,8.8,1398.0
97,Alexander,5.5,1390.0
98,The Last Airbender,4.2,1382.0


Let's also see how many movies in our dataset has less than 100 reviews.

In [ ]:
duckdb.query("""
SELECT COUNT(*)
FROM ratings_clean
WHERE num_user_for_reviews < 100""").to_df()

,count_star()
0,1759


As expected, there is a lot of movies with few reviews.

To remind our formula from the IMDb page:
weighted rating (WR) = (v ÷ (v+m)) x R + (m ÷ (v+m)) x C

Where:


R = mean rating for all titles

v = number of ratings for the title

m = minimum ratings required to be listed in the Top Rated 250 chart

C = the mean rating across the whole report

In [ ]:
mean_ratings = ratings_clean['imdb_score'].mean().round(2)
mean_ratings

np.float64(6.44)

It is important to point out that IMDb requires the minimum number of ratings to be 25,000, which is way more than even the most rated title in our dataset - The Lord of the Rings: The Fellowship of the Ring with	5060 reviews. So instead I'll set the value of m to the 80th percentile.

In [ ]:
m = ratings_clean['num_user_for_reviews'].quantile(0.80)
m

np.float64(383.2000000000003)

In [ ]:
weighted_rating = (ratings_clean['num_user_for_reviews']/(ratings_clean['num_user_for_reviews']+m))* ratings_clean['imdb_score'] + (m / (ratings_clean['num_user_for_reviews']+m)) * mean_ratings
weighted_rating

0       7.737230
1       6.943997
2       6.699832
3       8.244053
5       6.545316
          ...   
5038    6.459424
5039    6.952719
5040    6.438912
5041    6.436787
5042    6.468767
Length: 4900, dtype: float64

In [ ]:
ratings_clean['weighted_rating'] = weighted_rating
ratings_clean.head()

,movie_title,imdb_score,num_user_for_reviews,weighted_rating
0,Avatar,7.9,3054.0,7.737230
1,Pirates of the Caribbean: At World's End,7.1,1238.0,6.943997
2,Spectre,6.8,994.0,6.699832
3,The Dark Knight Rises,8.5,2701.0,8.244053
5,John Carter,6.6,738.0,6.545316


Now let's see the top 15 in sorted ranking based on the weighted rating.

In [ ]:
duckdb.query("""
SELECT *
FROM ratings_clean
ORDER BY weighted_rating DESC
limit 15 """).to_df()

,movie_title,imdb_score,num_user_for_reviews,weighted_rating
0,The Shawshank Redemption,9.3,4144.0,9.057918
1,The Dark Knight,9.0,4667.0,8.805752
2,The Godfather,9.2,2238.0,8.796508
3,The Lord of the Rings: The Return of the King,8.9,3189.0,8.636109
4,The Lord of the Rings: The Fellowship of the Ring,8.8,5060.0,8.633857
5,Pulp Fiction,8.9,2195.0,8.534368
6,Fight Club,8.8,2968.0,8.530141
7,Inception,8.8,2803.0,8.516166
8,The Matrix,8.7,3646.0,8.485061
9,The Lord of the Rings: The Two Towers,8.7,2417.0,8.390725


4. Exploring the budget dataset

In [ ]:
movie = pd.read_csv('data/budget.csv')
movie.columns

Index(['index', 'budget', 'genres', 'homepage', 'id', 'keywords',
       'original_language', 'original_title', 'overview', 'popularity',
       'production_companies', 'production_countries', 'release_date',
       'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title',
       'vote_average', 'vote_count', 'cast', 'crew', 'director'],
      dtype='object')

In [ ]:
movie.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

This dataset also has more columns than needed for this analysis.

In [ ]:
movies = movie[['title', 'budget', 'revenue']].copy()
movies['title'] = movies['title'].str.strip()
movies.head(10)

,title,budget,revenue
0,Avatar,237000000,2787965087
1,Pirates of the Caribbean: At World's End,300000000,961000000
2,Spectre,245000000,880674609
3,The Dark Knight Rises,250000000,1084939099
4,John Carter,260000000,284139100
5,Spider-Man 3,258000000,890871626
6,Tangled,260000000,591794936
7,Avengers: Age of Ultron,280000000,1405403694
8,Harry Potter and the Half-Blood Prince,250000000,933959197
9,Batman v Superman: Dawn of Justice,250000000,873260194


5. Cleaning the budget dataset

In [ ]:
movies.isnull().sum()

title      0
budget     0
revenue    0
dtype: int64

In [ ]:
movies.duplicated().sum()

np.int64(1)

In [ ]:
movies = movies.drop_duplicates()

It turns out that the dataset has no null values and only 1 duplicate row, but let's see if there are any movies with 0 budget or revenue.

In [ ]:
len(movies[(movies['budget']==0) | (movies['revenue']==0)])

1573

There is actually quite a bit, so let's get rid of those titles, since they won't be included in this ranking and will generate an error while trying to calculate the ROI later on.

In [ ]:
movies_clean = movies[(movies['budget']>0) & (movies['revenue']>0)].copy()

In [ ]:
movies_clean['revenue'].sort_values(ascending=False)

0       2787965087
25      1845034188
16      1519557910
28      1513528810
44      1506249360
           ...    
3131            11
2933            11
3875             7
3419             7
3372             5
Name: revenue, Length: 3229, dtype: int64

There are still some questionable values, like 5 dollar revenue, but let's see the outcome of the ROI calculation.

In [ ]:
movie_financial_success = (((movies_clean['revenue'] - movies_clean['budget'])/movies_clean['budget'])*100).round(2)
movie_financial_success.sort_values()

3875   -1.000000e+02
2874   -1.000000e+02
1655   -1.000000e+02
2068   -1.000000e+02
2485   -1.000000e+02
            ...     
4496    4.132333e+05
4582    5.329339e+05
4577    1.288939e+06
3137    9.999990e+07
4238    8.499999e+08
Length: 3229, dtype: float64

In [ ]:
movies_clean['movie_financial_success'] = movie_financial_success
movies_clean.sort_values('movie_financial_success', ascending=False).head(100)

,title,budget,revenue,movie_finanacial_success
4238,Modern Times,1,8500000,8.499999e+08
3137,Nurse 3-D,10,10000000,9.999990e+07
4577,Paranormal Activity,15000,193355800,1.288939e+06
4582,Tarnation,218,1162014,5.329339e+05
4496,The Blair Witch Project,60000,248000000,4.132333e+05
...,...,...,...,...
3825,Airplane!,3500000,83453539,2.284390e+03
4669,"The Beast from 20,000 Fathoms",210000,5000000,2.280950e+03
2514,Top Gun,15000000,356830601,2.278870e+03
2516,American Beauty,15000000,356296601,2.275310e+03


Above we can see top 100 movie titles based on financial success. Right away it is clear that there are issues with the dataset, as top position is being occupied by a movie with a budget of 1 dollar. The other positions look suspicious as well. Let's quickly compare it to top positions based only on revenue.

In [ ]:
movies.sort_values(['revenue'], ascending=False).head(100)

,title,budget,revenue
0,Avatar,237000000,2787965087
25,Titanic,200000000,1845034188
16,The Avengers,220000000,1519557910
28,Jurassic World,150000000,1513528810
44,Furious 7,190000000,1506249360
...,...,...,...
339,The Incredibles,92000000,631442092
270,The Martian,108000000,630161890
204,Fast Five,125000000,626137675
115,Hancock,150000000,624029371


From this perspective the data seems fine, since we see more familiar blockbuster titles, but the ROI revealed some hidden problems. Let's try to avoid some of those issues by filtering out the low budget and low revenue movies (in this case less than 100 000). 

In [ ]:
movies_cleaner = movies_clean[(movies_clean['budget']>=1000000) & (movies_clean['revenue']>=1000000)].copy()
financial_success = (((movies_cleaner['revenue'] - movies_cleaner['budget'])/movies_cleaner['budget'])*100).round(2)
movies_cleaner['financial_success'] = financial_success
movies_cleaner.sort_values('financial_success', ascending=False).head(100)

,title,budget,revenue,movie_finanacial_success,finanacial_success
4259,Snow White and the Seven Dwarfs,1488423,184925486,12324.26,12324.26
4333,Rocky,1000000,117235147,11623.51,11623.51
4347,The Devil Inside,1000000,101758490,10075.85,10075.85
3813,Gone with the Wind,4000000,400176459,9904.41,9904.41
4291,Saw,1200000,103911669,8559.31,8559.31
...,...,...,...,...,...
4282,Friday the 13th Part 2,1250000,21722776,1637.82,1637.82
3569,Paranormal Activity: The Marked Ones,5000000,86362372,1627.25,1627.25
3447,Butch Cassidy and the Sundance Kid,6000000,102308889,1605.15,1605.15
3448,Mary Poppins,6000000,102272727,1604.55,1604.55


The data still seems suspicious. Surely 1 dollar budget was an error, as well as 10 dollar one, but this undermines the validity of the whole dataset. At the same time what should be the cut off point for the minimum budget? Take, for example, The Blair Witch Project, which initially made the top five but was later cut out because of it's budget being lower than 100 000. It's a unique case of a very low-budget film achieving worldwide success, which potentially might have put it high on the ranking. Also is $60,000 the correct budget for The Blair Witch Project, or another mistake?

Because of this i decided to reach for another dataset from https://www.boxofficemojo.com/chart/top_lifetime_gross/?area=XWW which is generally considered to be an official source of movie related rankings. The main problem and a reason I didn't choose this dataset initially is that it lacks data about budget and has only data about lifetime gross. Because of this we can't actually see the inner workings of the financial success and have to adopt the lifetime gross as its metric. This also prevents us from finding out whether the budget of the movie tends to correlate with its critical success.

In [ ]:
gross = pd.read_excel('data/gross.ods', engine = 'odf')
gross

,rank,title,lifetime_gross,year
0,1,Avatar,"$2,923,710,708",2009
1,2,Avengers: Endgame,"$2,799,439,100",2019
2,3,Avatar: The Way of Water,"$2,334,484,620",2022
3,4,Titanic,"$2,264,812,968",1997
4,5,Ne Zha 2,"$2,259,822,417",2025
...,...,...,...,...
95,96,Guardians of the Galaxy Vol. 2,"$863,764,214",2017
96,97,Black Panther: Wakanda Forever,"$859,208,836",2022
97,98,Inside Out,"$859,076,254",2015
98,99,Venom,"$856,085,161",2018


First, let's check if there are any duplicates.

In [ ]:
gross.duplicated().sum()

np.int64(0)

In [ ]:
gross['title'].duplicated().sum()

np.int64(1)

In [ ]:
gross[gross['title'].duplicated()]

,rank,title,lifetime_gross,year
60,61,The Lion King,"$979,161,632",1994


In [ ]:
gross[gross['title']=='The Lion King']

,rank,title,lifetime_gross,year
11,12,The Lion King,"$1,662,020,819",2019
60,61,The Lion King,"$979,161,632",1994


It turns out that there is a duplicate title, but it did not appear by mistake. There are two Lion King movies, but one is the original 1994 animated movie and the other is 2019 remake. The fact that these two have the same title will cause problems later on when comparing to the table containing the critical success data. To avoid this, let's rename these titles based on the year of release and do the same for the second table.

In [ ]:
gross[gross['rank']==12] = gross[gross['rank']==12].replace({'The Lion King' : 'The Lion King (2019)'})

In [ ]:
gross[gross['rank']==61] = gross[gross['rank']==61].replace({'The Lion King' : 'The Lion King (1994)'})

Now we have to look into the original table with ratings to identify the Lion King versions there.

In [ ]:
rating_new = rating.copy()
rating_new.dropna()
rating_new['movie_title'] = rating_new['movie_title'].str.strip()
rating_new.drop_duplicates()
rating_new[['movie_title', 'title_year']][rating_new['movie_title']=='The Lion King']

,movie_title,title_year
509,The Lion King,1994.0


The Lion King in the ratings table is the original 1994 version, so let's rename it accordingly in the previously prepared 'ratings_clean' table that will be used in further analysis.

In [ ]:
ratings_clean[ratings_clean['movie_title']=='The Lion King'] = ratings_clean[ratings_clean['movie_title']=='The Lion King'].replace({'The Lion King' : 'The Lion King (1994)'})

Let's now join both tables containig the ranking based on critical success and lifetime gross ranking using SQL. This will find which of top 100 critically acclaimed movies appear also in top 100 lifetime gross ranking.

In [ ]:
overlap = duckdb.query("""
SELECT *
FROM ratings_clean
LEFT JOIN gross ON  ratings_clean.movie_title = gross.title
ORDER BY weighted_rating DESC
LIMIT 100
 """).to_df()

In [ ]:
overlap

,movie_title,imdb_score,num_user_for_reviews,weighted_rating,rank,title,lifetime_gross,year
0,The Shawshank Redemption,9.3,4144.0,9.057918,<NA>,None,None,<NA>
1,The Dark Knight,9.0,4667.0,8.805752,59,The Dark Knight,"$1,008,477,382",2008
2,The Godfather,9.2,2238.0,8.796508,<NA>,None,None,<NA>
3,The Lord of the Rings: The Return of the King,8.9,3189.0,8.636109,34,The Lord of the Rings: The Return of the King,"$1,148,812,312",2003
4,The Lord of the Rings: The Fellowship of the Ring,8.8,5060.0,8.633857,83,The Lord of the Rings: The Fellowship of the Ring,"$896,931,290",2001
...,...,...,...,...,...,...,...,...
95,Finding Nemo,8.2,866.0,7.660109,75,Finding Nemo,"$941,637,960",2003
96,Deadpool,8.1,1058.0,7.658623,<NA>,None,None,<NA>
97,Snatch,8.3,726.0,7.657418,<NA>,None,None,<NA>
98,The Martian,8.1,1023.0,7.647638,<NA>,None,None,<NA>


 What is important here is that we can see some null values, which means that these movie titles on the left from the from critical success table didn't appear in top 100 lifetime gross ranking - no match was found for those titles in the other table. Notably number 1 top rated The Shawshank Redemption and The Godfather on the 3rd place didn't make the financial success ranking. Let's see which other top reviewed movies didn't make it.

In [ ]:
overlap[overlap['rank'].isnull()]

,movie_title,imdb_score,num_user_for_reviews,weighted_rating,rank,title,lifetime_gross,year
0,The Shawshank Redemption,9.3,4144.0,9.057918,<NA>,None,None,<NA>
2,The Godfather,9.2,2238.0,8.796508,<NA>,None,None,<NA>
5,Pulp Fiction,8.9,2195.0,8.534368,<NA>,None,None,<NA>
6,Fight Club,8.8,2968.0,8.530141,<NA>,None,None,<NA>
7,Inception,8.8,2803.0,8.516166,<NA>,None,None,<NA>
...,...,...,...,...,...,...,...,...
93,Gran Torino,8.2,871.0,7.662261,<NA>,None,None,<NA>
96,Deadpool,8.1,1058.0,7.658623,<NA>,None,None,<NA>
97,Snatch,8.3,726.0,7.657418,<NA>,None,None,<NA>
98,The Martian,8.1,1023.0,7.647638,<NA>,None,None,<NA>


This is surprising since 88 top reviewed movies out top 100 didn't appear on the top 100 lifetime gross ranking.

In [ ]:
overlap_clean = overlap.dropna()
overlap_clean

,movie_title,imdb_score,num_user_for_reviews,weighted_rating,rank,title,lifetime_gross,year
1,The Dark Knight,9.0,4667.0,8.805752,59,The Dark Knight,"$1,008,477,382",2008
3,The Lord of the Rings: The Return of the King,8.9,3189.0,8.636109,34,The Lord of the Rings: The Return of the King,"$1,148,812,312",2003
4,The Lord of the Rings: The Fellowship of the Ring,8.8,5060.0,8.633857,83,The Lord of the Rings: The Fellowship of the Ring,"$896,931,290",2001
9,The Lord of the Rings: The Two Towers,8.7,2417.0,8.390725,73,The Lord of the Rings: The Two Towers,"$944,677,423",2002
14,The Dark Knight Rises,8.5,2701.0,8.244053,41,The Dark Knight Rises,"$1,085,429,532",2012
63,The Avengers,8.1,1722.0,7.797838,13,The Avengers,"$1,520,538,536",2012
72,The Lion King (1994),8.5,656.0,7.740385,61,The Lion King (1994),"$979,161,632",1994
74,Avatar,7.9,3054.0,7.737230,1,Avatar,"$2,923,710,708",2009
81,Captain America: Civil War,8.2,1022.0,7.720046,32,Captain America: Civil War,"$1,155,046,416",2016
85,Inside Out,8.3,773.0,7.683539,98,Inside Out,"$859,076,254",2015


In [ ]:
f'Number of overlaping rows: {overlap_clean.index.value_counts().sum()}'

'Number of overlaping rows: 12'

As we can see, only the The Lion King (1994) title appears in our table. Without the previos editing of the titles, we would have two Lion King versions joined to the one on the left, giving us the extra row.

In [ ]:
overlaptop50 = overlap_clean[overlap_clean['rank']<=50]
overlaptop50

,movie_title,imdb_score,num_user_for_reviews,weighted_rating,rank,title,lifetime_gross,year
3,The Lord of the Rings: The Return of the King,8.9,3189.0,8.636109,34,The Lord of the Rings: The Return of the King,"$1,148,812,312",2003
14,The Dark Knight Rises,8.5,2701.0,8.244053,41,The Dark Knight Rises,"$1,085,429,532",2012
63,The Avengers,8.1,1722.0,7.797838,13,The Avengers,"$1,520,538,536",2012
74,Avatar,7.9,3054.0,7.737230,1,Avatar,"$2,923,710,708",2009
81,Captain America: Civil War,8.2,1022.0,7.720046,32,Captain America: Civil War,"$1,155,046,416",2016
94,Toy Story 3,8.3,733.0,7.661448,45,Toy Story 3,"$1,067,316,101",2010


In the top 100 rated movies and top 100 grossing movies only 12 overlap and in top 50 of total gross there is only only 6, meaning that large majority of critically acclaimed and widely popular movies are not as financially successful. It is quite interesting outcome since one might think that these two would have stronger correlation. This then opens a further questions about the reasons for that divergence. Fully answering those questions goes beyond this analysis. At this point I could only propose two possible reasons for the outcome of the analysis. One simply being inflation. This is a crucial limitation of this analysis, since the financial data is not adjusted for inflation, and therefore we cannot preciselly establish that rank of the given movie. Secondly, we could speculate about the effectiveness of the movie marketing, accounting for a financial success, but not necessarily critical success. Lots of people might be persuaded to see given movie by it's marketing, which doesn't mean they will automatically like the movie. Most likely, these two causes occur simultaneously, and there are probably many others. Despite those limitations, the finding remain intriguing and could serve as a ground for further analysis, adopting the conclusion of this one as it's main research question, namely: 'Why there is a divergence between a critical success and the financial success in the movie industry?'